# Exploratory Data Analysis: Loan Approval Prediction

This notebook explores the Kaggle / Analytics Vidhya Loan Prediction Dataset (614 records, 13 attributes).
It establishes data distributions, missing value patterns, outlier behavior, and feature correlations with the target `Loan_Status`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/raw/train.csv")
print("Dataset Shape:", df.shape)
df.head()

### Initial Shape & Structure Analysis
The dataset consists of 614 applicant records with 12 features and 1 binary target (`Loan_Status`).
Features include demographic attributes (Gender, Married, Dependents, Education, Self_Employed), financial attributes (ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History), and geographical location (Property_Area).

In [ ]:
# Missing Values and Data Types
missing_df = pd.DataFrame({
    'Dtype': df.dtypes,
    'Missing_Count': df.isna().sum(),
    'Missing_Pct': (df.isna().sum() / len(df) * 100).round(2)
})
missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

### Missingness Findings
Seven features have missing values:
- `Credit_History`: 50 missing values (8.14%) - crucial predictor; missing values need mode imputation (1.0).
- `Self_Employed`: 32 missing values (5.21%) - categorical; impute with mode ('No').
- `LoanAmount`: 22 missing values (3.58%) - continuous; impute with median due to right-skew.
- `Dependents`: 15 missing values (2.44%) - contains '3+' strings requiring normalization.
- `Loan_Amount_Term`: 14 missing values (2.28%) - impute with median (360 months).
- `Gender` (13) and `Married` (3) - sensitive attributes; not used as model inputs but imputed for audit completeness.

In [ ]:
# Target Variable Distribution
status_counts = df['Loan_Status'].value_counts()
status_pct = df['Loan_Status'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
status_counts.plot(kind='bar', color=['#2ca02c', '#d62728'], ax=ax[0])
ax[0].set_title('Loan Status Count')
ax[0].set_xlabel('Loan Status (Y=Approved, N=Rejected)')
ax[0].set_ylabel('Count')

ax[1].pie(status_counts, labels=['Approved (Y)', 'Rejected (N)'], autopct='%1.1f%%', colors=['#2ca02c', '#d62728'], startangle=140)
ax[1].set_title('Loan Approval Ratio')
plt.tight_layout()
plt.show()
print(f"Approved (Y): {status_counts['Y']} ({status_pct['Y']:.1f}%)")
print(f"Rejected (N): {status_counts['N']} ({status_pct['N']:.1f}%)")

### Target Balance Analysis
There is moderate class imbalance in the target variable:
- 68.7% Approved (Y = 422)
- 31.3% Rejected (N = 192)

Because of this roughly 2:1 class ratio, naive models predicting all approvals would achieve ~68.7% accuracy. Therefore, evaluation must focus on Stratified 5-Fold Cross-Validation, F1-Score, Precision-Recall balance, and ROC-AUC, alongside class-weighted or SMOTE balancing techniques.

In [ ]:
# Income and LoanAmount Distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['ApplicantIncome'], kde=True, ax=axes[0, 0], color='navy')
axes[0, 0].set_title('ApplicantIncome Distribution')

sns.boxplot(x=df['ApplicantIncome'], ax=axes[0, 1], color='lightblue')
axes[0, 1].set_title('ApplicantIncome Boxplot (Outliers)')

sns.histplot(df['LoanAmount'].dropna(), kde=True, ax=axes[1, 0], color='green')
axes[1, 0].set_title('LoanAmount Distribution')

sns.boxplot(x=df['LoanAmount'].dropna(), ax=axes[1, 1], color='lightgreen')
axes[1, 1].set_title('LoanAmount Boxplot (Outliers)')

plt.tight_layout()
plt.show()

### Distribution & Outlier Insights
- Both `ApplicantIncome` and `LoanAmount` are heavily right-skewed with significant high-income and high-loan outliers.
- Maximum `ApplicantIncome` is 81,000 against a median of 3,812.5.
- Maximum `LoanAmount` is 700 against a median of 128.0.
- Strategy: Apply IQR outlier-capping and create `Log_Total_Income = log1p(Total_Income)` to stabilize variance and prevent extreme outliers from distorting linear and distance-based estimators.

In [ ]:
# Approval Rate by Key Features: Credit_History, Education, Property_Area
features_to_check = ['Credit_History', 'Education', 'Property_Area']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(features_to_check):
    cross_tab = pd.crosstab(df[col], df['Loan_Status'], normalize='index') * 100
    cross_tab.plot(kind='bar', stacked=True, color=['#d62728', '#2ca02c'], ax=axes[i])
    axes[i].set_title(f'Approval Rate by {col}')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].legend(['Rejected (N)', 'Approved (Y)'], loc='upper right')

plt.tight_layout()
plt.show()

### Approval Rate Drivers
1. **Credit History is the dominant signal**:
   - Applicants with `Credit_History == 1.0` have an approval rate of ~79.6%.
   - Applicants with `Credit_History == 0.0` have an approval rate of only ~8.0%.
2. **Education**:
   - Graduates have a ~70.8% approval rate vs ~61.2% for Non-Graduates.
3. **Property Area**:
   - `Semiurban` properties have the highest approval rate (~76.8%), followed by `Urban` (~65.8%) and `Rural` (~61.5%).

In [ ]:
# Correlation Heatmap of Numeric Features
df_numeric = df.select_dtypes(include=[np.number]).copy()
df_numeric['Target_Numeric'] = (df['Loan_Status'] == 'Y').astype(int)

corr = df_numeric.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features & Target')
plt.show()

### Correlation Summary
- `Credit_History` exhibits the strongest positive correlation with `Target_Numeric` (~0.56).
- `LoanAmount` correlates strongly with `ApplicantIncome` (~0.57) and `CoapplicantIncome` (~0.19), validating that higher earners request larger loans.
- Engineered features like `Total_Income`, `EMI`, and `Loan_to_Income` will help models separate affordability and repayment capacity more effectively than raw income alone.